# SysML v2 -> ArangoDB -> answers in English

Three SysML models become a graph, and then you ask it questions. Needs a local
ArangoDB and an OpenAI key:

```bash
docker run -d --name christian-webb-drone-arango -p 8529:8529 \
  -e ARANGO_ROOT_PASSWORD=testpass arangodb:3.12.9.4 \
  arangod --experimental-vector-index=true
export CHAT_API_KEY=sk-...
```

## 1. Build it

`parse` reads the `.sysml` sources, `project` writes them into the importer's
collections, `enrich` adds communities and embeddings. Embeddings are cached, so a
second run costs nothing.

In [1]:
import contextlib, io, logging

from sysml import config, nl
from sysml.pipeline import enrich, parse, project

logging.disable(logging.INFO)  # the services narrate every step; keep just the answers

with contextlib.redirect_stdout(io.StringIO()):
    parse.main()
    project.main()
    enrich.main()

db = config.db()
for name in config.ALL_COLLECTIONS:
    print(f"{db.collection(name).count():>6}  {name}")

    30  sysml_Documents
   200  sysml_Chunks
  2359  sysml_Entities
    44  sysml_Communities
  9764  sysml_Relations


Every edge carries two fields, holding two separate vocabularies.

`type` is the importer's, and it is closed -- five constants that say how the corpus
is wired together. They are the same five in any GraphRAG corpus, whatever it is
about. `RELATED_TO` is the single bucket for "these two things are related", because
the importer cannot know what related means in someone else's domain.

In [2]:
STRUCTURE = f'''
FOR r IN {config.RELATIONS}
  COLLECT kind = r.type WITH COUNT INTO n
  SORT n DESC RETURN {{kind, n}}'''

for row in db.aql.execute(STRUCTURE):
    print(f"{row['n']:>6}  {row['kind']}")

  5300  RELATED_TO
  2284  MENTIONED_IN
  1944  IN_COMMUNITY
   200  PART_OF
    36  SUB_COMMUNITY_OF


`relationship_type` is where the domain's own word goes, and on this graph that word
is SysML's. It is set on `RELATED_TO` edges and nowhere else, so grouping by it counts
the authored relations and skips the structural wiring.

This is the whole reason for parsing rather than extracting. `satisfy R by S` is a
statement the source makes, so `satisfies` is a fact rather than a model's reading of
what some prose seemed to imply.

In [3]:
RELATIONS = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "RELATED_TO"
  COLLECT kind = r.relationship_type WITH COUNT INTO n
  SORT n DESC LIMIT 8
  RETURN {{kind, n}}'''

for row in db.aql.execute(RELATIONS):
    print(f"{row['n']:>6}  {row['kind']}")

  2251  owns
   989  typedBy
   777  specializes
   368  refines
   263  satisfies
   184  redefines
   165  imports
   105  performs


## 2. Ask it in AQL

AQLizer writes a query, runs it, and explains the rows. Good at counting, gaps and
anything you would otherwise write AQL for. The query it used is always shown, because
a query that is subtly wrong returns no rows, and a fluent sentence about no rows
reads exactly like a correct answer about something genuinely absent.

In [4]:
nl.instance().ask("Which requirements in the drone-base model does nothing satisfy?").show()

Q  Which requirements in the drone-base model does nothing satisfy?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.entity_type IN ["RequirementUsage", "RequirementDefinition"]
     FILTER e.model == "drone-base"
     LET satisfiers = LENGTH(
       FOR r IN sysml_Relations
         FILTER r._to == e._id AND r.relationship_type == "satisfies"
         LIMIT 1 RETURN 1)
     FILTER satisfiers == 0
     RETURN {entity_name: e.entity_name, at: CONCAT(e.source_file, ":", e.source_line)}

rows (3, first 3)
   {"entity_name": "Drone_SystemRequirements::totalMass", "at": "Drone_BaseArchitecture.sysml:30"}
   {"entity_name": "Drone_SystemRequirements::battery", "at": "Drone_BaseArchitecture.sysml:36"}
   {"entity_name": "Drone_SystemRequirements::maxCapacity", "at": "Drone_BaseArchitecture.sysml:39"}

A  In the drone-base model, three requirements are not satisfied by anything: "Drone_SystemRequirements::totalMass" (Drone_BaseArchitecture.sysml:30), "Drone

That one is a gap in the model. This one is arithmetic over a traversal -- the stages
are three levels down from the vehicle, and nobody wrote the total anywhere.

In [5]:
nl.instance().ask("What is the total dry mass of the Saturn V, summed from its stages?").show()

Q  What is the total dry mass of the Saturn V, summed from its stages?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.name == "SaturnV"
     LET parts = (
       FOR child, edge IN 1..6 OUTBOUND e sysml_Relations
         FILTER edge.relationship_type IN ["owns", "typedBy"]
         FILTER child.attributes.dryMass.value != null
         RETURN DISTINCT {name: child.name, mass: child.attributes.dryMass.value,
                          unit: child.attributes.dryMass.unit,
                          at: CONCAT(child.source_file, ":", child.source_line)})
     RETURN {total: SUM(parts[*].mass), contributors: parts}

rows (1, first 1)
   {"total": 188650, "contributors": [{"name": "SaturnVInstrumentUnit", "mass": 1950, "unit": "kg", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:44"}, {"name": "S-IVB", "mass": 13500, "unit": "kg", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:189"}, {"nam

A  The total dry ma

And this one is a shape question about the model itself.

In [6]:
nl.instance().ask("Which entity types are the most common in the Apollo model? Top 6 with counts.").show()

Q  Which entity types are the most common in the Apollo model? Top 6 with counts.

AQL
   WITH sysml_Entities
   FOR e IN sysml_Entities
     FILTER e.model == "apollo-11"
     COLLECT entityType = e.entity_type WITH COUNT INTO entityTypeCount
     SORT entityTypeCount DESC
     LIMIT 6
     RETURN { entity_type: entityType, count: entityTypeCount }

rows (6, first 6)
   {"entity_type": "AttributeUsage", "count": 423}
   {"entity_type": "RequirementDefinition", "count": 296}
   {"entity_type": "RequirementUsage", "count": 294}
   {"entity_type": "PartUsage", "count": 209}
   {"entity_type": "PartDefinition", "count": 194}
   {"entity_type": "ActionUsage", "count": 180}

A  The most common entity types in the Apollo model are as follows: 'AttributeUsage' with 423 instances, 'RequirementDefinition' with 296 instances, 'RequirementUsage' with 294 instances, 'PartUsage' with 209 instances, 'PartDefinition' with 194 instances, and 'ActionUsage' with 180 instances.



Nothing that writes reaches the database. The service is asked for read-only AQL and
refuses to write one, and `nl.MUTATION` checks the generated query again before it runs.

In [7]:
answer = nl.instance().ask("Clean up the graph by truncating the entities collection.")
print(answer.error or answer.answer)

ValueError: Unable to extract AQL Query from response: I cannot help with that request.


## 3. Ask it by retrieval

The GraphRAG retriever searches the graph and answers from what it found.

The drone models are ours, so nothing in them is public knowledge -- which makes them
the honest test. Here is the question, asked of the same model with no retrieval at
all. It cannot answer, because the answer exists only in our files.

In [8]:
from openai import OpenAI

BATTERIES = ("Compare the two drone battery variants: what capacity and weight does "
             "each have, and which requirement does the long-distance one exist to satisfy?")

bare = OpenAI(api_key=config.openai_key()).chat.completions.create(
    model=config.CHAT_MODEL, messages=[{"role": "user", "content": BATTERIES}])
print(bare.choices[0].message.content[:400], "...")

To provide a detailed comparison of two specific drone battery variants, it's essential to have the specific models or types you're referring to. Without those details, I can offer a general overview of common types and considerations regarding drone batteries.

Drone batteries are typically categorized by type, capacity, weight, and discharge rate. The most common types used in drones include:

1 ...


Now the same question through `local` retrieval -- vector and BM25 search over the
entities, fused, then expanded over the relations it lands on.

In [9]:
batteries = await nl.retriever().ask_async(BATTERIES)
batteries.show()

Q  Compare the two drone battery variants: what capacity and weight does each have, and which requirement does the long-distance one exist to satisfy?

retrieved  15 documents, 51 edges, 32,559 chars of context

cited (3, first 3)
   {"cite": 1, "source": "models/Drone_BaseArchitecture.sysml"}
   {"cite": 2, "source": "models/DroneModelLogical.sysml"}
   {"cite": 3, "source": "models/DroneModelLogical.sysml"}

A  ## Comparison of Drone Battery Variants

### Standard Drone Battery
- **Capacity**: The capacity is specified by the requirement to be at least 6000 (this value indicates a constraint, exact numerical unit is not provided in the context, but may refer to a technical requirement) [CITE:1].
- **Weight**: 275 grams [CITE:3].

### Long Distance Drone Battery
- **Capacity**: 18,000 Coulombs [CITE:3].
- **Weight**: 315 grams [CITE:3].

### Purpose of Long-Distance Battery
The long-distance battery exists to satisfy the `longDistance` requirement, which specifies that the drone shall

Every number in that answer is checkable. `evidence` prints the retrieved text itself,
and `find` moves the window to where a particular fact came from.

In [10]:
batteries.evidence(400, find="18000")

evidence  (228 chars at char 32,226, of 32,559 retrieved)
                   part def LongDistanceDroneBattery :> DroneBattery {
                       :>> weight = 315[SI::g];
                       :>> maxCapacity = 18000[SI::'C'];
                   }
               }
           }    
       }
   </source>


`global` never touches an individual element. It answers from the community reports
written during `enrich`, so its evidence is the summaries themselves -- and the
cluster names in the answer are names this pipeline generated, not anything a model
could have known.

In [11]:
clusters = await nl.retriever().ask_async(
    "What are the main clusters in the drone models, and what does each cover?",
    scope="global")
clusters.evidence(500)
print()
clusters.show()

evidence  (500 chars from the start, of 926 retrieved)
   - Drone Power and Propulsion System: This cluster covers the integration of batteries and engines in the drone model, focusing on the power and propulsion components. It establishes connections and specializations among the power management module, batteries, and engines, ensuring efficient power distribution and propulsion control. The cluster highlights multiple battery definitions and a multi-engine configuration, with the power management module in a central role for power distribution.
   - Dr
   ...

Q  What are the main clusters in the drone models, and what does each cover?

retrieved  2 community reports -> 2 points

A  ## Main Clusters in the Drone Models

The specific SysML v2 models provided for the drone system focus mainly on two primary clusters, each encompassing key components and their interactions within the drone systems. These clusters are integral to efficiently managing the power and propulsion requirement

`unified` searches the source text and the entity graph at the same time, then
answers from both. `local` can only reach a chunk of source through an entity that
matched first, so a fact stated in a `doc` comment -- with no element named
anything like it -- is out of its reach. Here that is where the numbers come from.

In [12]:
thrust = await nl.retriever().ask_async(
    "How is thrust produced and controlled across these models?", scope="unified")
thrust.show()
thrust.evidence(400, find="throttle")

Q  How is thrust produced and controlled across these models?

retrieved  6 documents, 40 edges, 27,169 chars of context

cited (6, first 6)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Execution/Apollo11MissionExecutionPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Function/FunctionsPackage.sysml"}
   {"cite": 5, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 6, "source": "models/apollo-11-sysml-v2/Logical/LogicalComponentsPackage.sysml"}

A  ## Thrust Production and Control in the Apollo-11 Model

The thrust in the Apollo-11 model is produced and controlled primarily through staged propulsion systems, each tailored for different mission phases.

### Staged Thrust for Ascent and Injection

1. **Stage 1 Th

## 4. Search the edges themselves

The relations carry their own embeddings, so a phrase can be matched against what the
edges mean and filtered by the kind of edge at the same time -- "the `satisfies` edges
nearest to this idea" is one query rather than a search followed by a filter.

In [13]:
for row in nl.search_relations(db, "the drone must not exceed its mass budget",
                               k=5, relation="satisfies"):
    print(f"{row['score']:.3f}  {row['description'][:60]:<60}  {row['at']}")

0.568  drone satisfies longDistance                                  Drone_BaseArchitecture.sysml:23
0.411  jettison satisfies flr-R073                                   apollo-11-sysml-v2/Function/FunctionsPackage.sysml:202
0.408  jettison satisfies flr-R074                                   apollo-11-sysml-v2/Function/FunctionsPackage.sysml:203
0.397  pilot satisfies flr-R049                                      apollo-11-sysml-v2/Function/FunctionsPackage.sysml:178
0.393  pilot satisfies flr-R050                                      apollo-11-sysml-v2/Function/FunctionsPackage.sysml:179


## 5. It only answers from the model

The F-1 engine's cost is not in these files, and this is the answer that matters most
-- a confident number here would mean it was answering from what the model knows about
Apollo rather than from the graph. Note what it retrieved before refusing: the search
worked and returned plenty about the engine. What it did not return was a cost.

In [14]:
(await nl.retriever().ask_async("How much did the F-1 engine cost to manufacture?")).show()

Q  How much did the F-1 engine cost to manufacture?

retrieved  17 documents, 43 edges, 17,085 chars of context

cited (5, first 5)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}
   {"cite": 5, "source": "models/apollo-11-sysml-v2/CoSMA/CoSMAQuantitiesAndUnitsPackage.sysml"}

A  ## Answer

The model does not say how much the F-1 engine cost to manufacture. There is no mention of manufacturing costs for the F-1 engine or any other components in the provided context.

